In [1]:
!pip install torch torchvision pyhealth

  Using cached pandas-2.3.3-cp312-cp312-manylinux_2_24_x86_64.manylinux_2_28_x86_64.whl.metadata (91 kB)
  Using cached pyarrow-22.0.0-cp312-cp312-manylinux_2_28_x86_64.whl.metadata (3.2 kB)
Using cached pandas-2.3.3-cp312-cp312-manylinux_2_24_x86_64.manylinux_2_28_x86_64.whl (12.4 MB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 783.6/783.6 kB 46.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.3/41.3 MB 61.5 MB/s eta 0:00:00:00:0100:01
Using cached pyarrow-22.0.0-cp312-cp312-manylinux_2_28_x86_64.whl (47.7 MB)
  Attempting uninstall: pyarrow
    Found existing installation: pyarrow 24.0.0
    Uninstalling pyarrow-24.0.0:
      Successfully uninstalled pyarrow-24.0.0
  Attempting uninstall: polars-runtime-32
    Found existing installation: polars-runtime-32 1.40.0
    Uninstalling polars-runtime-32-1.40.0:
      Successfully uninstalled polars-runtime-32-1.40.0
  Attempting uninstall: polars
    Found existing installation: polars 1.40.0
    Uninstalling polars-1.40.0

In [2]:
!pip install pyhealth

In [3]:
!pip install --upgrade pyarrow pandas polars

  Using cached pyarrow-24.0.0-cp312-cp312-manylinux_2_28_x86_64.whl.metadata (3.0 kB)
  Using cached pandas-3.0.2-cp312-cp312-manylinux_2_24_x86_64.manylinux_2_28_x86_64.whl.metadata (79 kB)
  Using cached polars-1.40.0-py3-none-any.whl.metadata (10 kB)
  Using cached polars_runtime_32-1.40.0-cp310-abi3-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (1.5 kB)
Using cached pyarrow-24.0.0-cp312-cp312-manylinux_2_28_x86_64.whl (48.9 MB)
Using cached pandas-3.0.2-cp312-cp312-manylinux_2_24_x86_64.manylinux_2_28_x86_64.whl (10.9 MB)
Using cached polars-1.40.0-py3-none-any.whl (828 kB)
Using cached polars_runtime_32-1.40.0-cp310-abi3-manylinux_2_17_x86_64.manylinux2014_x86_64.whl (56.2 MB)
  Attempting uninstall: pyarrow
    Found existing installation: pyarrow 22.0.0
    Uninstalling pyarrow-22.0.0:
      Successfully uninstalled pyarrow-22.0.0
  Attempting uninstall: polars-runtime-32
    Found existing installation: polars-runtime-32 1.35.2
    Uninstalling polars-runtime-32-1.35.

In [ ]:
import os
import torch
import pandas as pd
import torch.nn as nn
from typing import Dict, List, Optional
from torch.utils.data import DataLoader
from torchvision import transforms
from transformers import ViTModel, BertModel, AutoTokenizer

from pyhealth.datasets import BaseDataset
from pyhealth.tasks import BaseTask
from pyhealth.models import BaseModel
from pyhealth.trainer import Trainer

# Set device
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

Using device: cuda


In [ ]:
def setup_synthetic_environment(root="./synthetic_mimic"):
    """
    Generates a minimal, functional dataset for logic verification.
    
    RATIONALE FOR SYNTHETIC DATA:
    Using synthetic data instead of pre-existing datasets (like full MIMIC-CXR) 
    is essential for development and Pull Request validation. It allows for:
    1. Instant testing without 500GB+ storage requirements.
    2. Compliance with privacy standards by avoiding Protected Health Information (PHI).
    3. Automated CI/CD compatibility, ensuring code remains performant and 
       reproducible across different environments without credentialed access.
    """
    os.makedirs(os.path.join(root), exist_ok=True)
    os.makedirs("./configs", exist_ok=True)
    
    # 1. Metadata: Links patients (subject_id) to studies and images
    pd.DataFrame({
        "dicom_id": [f"d{i}" for i in range(12)],
        "subject_id": ["p1"]*6 + ["p2"]*6,
        "study_id": [f"s{i}" for i in range(12)],
        "study_date": [f"20260{i+1:02d}01" for i in range(12)]
    }).to_csv(os.path.join(root, "mimic-cxr-2.1.0-metadata.csv.gz"), index=False)

    # 2. Labels: Multi-label diagnosis
    pd.DataFrame({
        "dicom_id": [f"d{i}" for i in range(12)],
        "Atelectasis": [1, 0, 1, 0, 1, 0] * 2,
        "Cardiomegaly": [0, 1, 0, 1, 0, 1] * 2
    }).to_csv(os.path.join(root, "mimic-cxr-2.1.0-chexpert.csv.gz"), index=False)

    # 3. Reports: Historical text context
    pd.DataFrame({
        "study_id": [f"s{i}" for i in range(12)],
        "report_text": ["Normal findings."] * 4 + ["Developing opacity."] * 4 + ["Stable condition."] * 4
    }).to_csv(os.path.join(root, "mimic-cxr-sections.csv.gz"), index=False)
    
    # 4. YAML Config: Schema for PyHealth data joins
    with open("./configs/mimic_cxr_longitudinal.yaml", "w") as f:
        f.write("""
dataset_name: "MIMIC-CXR-Longitudinal"
tables:
  metadata:
    file_name: "mimic-cxr-2.1.0-metadata.csv.gz"
    primary_key: "dicom_id"
    patient_key: "subject_id"
    visit_key: "study_id"
    timestamp_key: "study_date"
  chexpert:
    file_name: "mimic-cxr-2.1.0-chexpert.csv.gz"
    primary_key: "dicom_id"
  reports:
    file_name: "mimic-cxr-sections.csv.gz"
    primary_key: "study_id"
        """)
    print("Synthetic environment and YAML configuration initialized.")

setup_synthetic_environment()

Synthetic environment and YAML configuration initialized.


In [ ]:
class MIMICCXRLongitudinalDataset(BaseDataset):
    def __init__(self, root, config_path="./configs/mimic_cxr_longitudinal.yaml", **kwargs):
        super().__init__(root=root, config_path=config_path, **kwargs)

    def parse_event(self, table_name, row):
        return {k.lower(): v for k, v in row.items()}

class HistAIDTask(BaseTask):
    def __init__(self, max_history=3, **kwargs):
        super().__init__(**kwargs)
        self.max_history = max_history

    def __call__(self, patient):
        samples = []
        report_history = []
        # Ensure visits are sorted for longitudinal analysis
        visits = sorted(patient.visits, key=lambda v: v.encounter_time)
        for visit in visits:
            metadata = visit.get_event_by_table("metadata")
            labels = visit.get_event_by_table("chexpert")
            if metadata and labels:
                samples.append({
                    "patient_id": patient.patient_id,
                    "image": metadata[0]["dicom_id"],
                    "history": list(report_history),
                    "label": [float(labels[0]["atelectasis"]), float(labels[0]["cardiomegaly"])]
                })
            report = visit.get_event_by_table("reports")
            if report:
                report_history.append(report[0]["report_text"])
                report_history = report_history[-self.max_history:]
        return samples

In [ ]:
class HistAID(BaseModel):
    def __init__(self, dataset, fusion_dim=512, **kwargs):
        super().__init__(dataset=dataset, **kwargs)
        # Vision Stream: Global context via ViT
        self.vision_encoder = ViTModel.from_pretrained("google/vit-base-patch16-224-in21k")
        # Text Stream: Semantic history via BERT
        self.text_encoder = BertModel.from_pretrained("bert-base-uncased")
        
        self.vision_proj = nn.Linear(768, fusion_dim)
        self.text_proj = nn.Linear(768, fusion_dim)
        
        # Transformer Fusion: Learns cross-modal dependencies
        self.fusion_transformer = nn.TransformerEncoderLayer(
            d_model=fusion_dim, nhead=8, batch_first=True
        )
        self.classifier = nn.Linear(fusion_dim, self.num_labels)

    def forward(self, image, history_input_ids, history_attention_mask, label, **kwargs):
        # 1. Vision Embedding
        v_out = self.vision_encoder(pixel_values=image).last_hidden_state[:, 0, :]
        v_token = self.vision_proj(v_out).unsqueeze(1) 

        # 2. Text/Temporal Embedding
        b, s, l = history_input_ids.shape
        t_out = self.text_encoder(history_input_ids.view(-1, l), history_attention_mask.view(-1, l))
        t_feat = t_out.last_hidden_state[:, 0, :].view(b, s, -1).mean(dim=1)
        t_token = self.text_proj(t_feat).unsqueeze(1)

        # 3. Transformer Fusion (Token-based Interaction)
        fused = self.fusion_transformer(torch.cat([v_token, t_token], dim=1)).mean(dim=1)
        
        logits = self.classifier(fused)
        y_prob = torch.sigmoid(logits)
        loss = nn.BCEWithLogitsLoss()(logits, label.float())
        return {"loss": loss, "y_prob": y_prob, "y_true": label}

In [ ]:
# batching and tokenization
tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")

def multi_modal_collate(batch):
    # Mock image loading (3-channel random noise for logic check)
    images = torch.randn(len(batch), 3, 224, 224) 
    labels = torch.stack([torch.tensor(b["label"]) for b in batch])
    
    # Process history: take the most recent report or a pad string
    histories = [b["history"][-1] if b["history"] else "No history" for b in batch]
    tokens = tokenizer(histories, padding="max_length", max_length=64, return_tensors="pt")
    
    return {
        "image": images,
        "history_input_ids": tokens["input_ids"].unsqueeze(1),
        "history_attention_mask": tokens["attention_mask"].unsqueeze(1),
        "label": labels
    }

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:104: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

In [ ]:
ablation_results = []

# Experiment 1 & 2: Dataset Feature (History vs No History) 
# and Model Capacity (Fusion Dim)
configs = [
    {"max_history": 0, "fusion_dim": 512, "lr": 1e-4, "tag": "No History (Baseline)"},
    {"max_history": 3, "fusion_dim": 512, "lr": 1e-4, "tag": "With History (Longitudinal)"},
    {"max_history": 3, "fusion_dim": 256, "lr": 1e-4, "tag": "Small Fusion Dim"},
]

for config in configs:
    print(f"\n>>> Running Ablation: {config['tag']}")
    dataset = MIMICCXRLongitudinalDataset(root="./synthetic_mimic")
    task = HistAIDTask(max_history=config['max_history'])
    samples = dataset.set_task(task)
    
    model = HistAID(dataset=samples, fusion_dim=config['fusion_dim']).to(device)
    loader = DataLoader(samples, batch_size=2, collate_fn=multi_modal_collate)
    
    trainer = Trainer(model=model, metrics=["roc_auc_samples", "pr_auc_samples"])
    trainer.train(
        train_dataloader=loader, 
        val_dataloader=loader, 
        epochs=3, 
        optimizer_params={"lr": config['lr']}
    )
    
    ablation_results.append({
        "Configuration": config['tag'],
        "AUROC": trainer.best_val_stats["roc_auc_samples"],
        "AUPRC": trainer.best_val_stats["pr_auc_samples"]
    })

# Display Performance Comparison
print("\n" + "="*50)
print("ABLATION STUDY PERFORMANCE COMPARISON")
print("="*50)
print(pd.DataFrame(ablation_results))


>>> Running Ablation: No History (Baseline)


NameError: name 'MIMICCXRLongitudinalDataset' is not defined